In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

## Obteniendo los parámetros

In [0]:
%sql
CREATE WIDGET TEXT catalog_name DEFAULT 'smartdata_project';
CREATE WIDGET TEXT schema_source DEFAULT 'bronze';
CREATE WIDGET TEXT schema_sink DEFAULT 'silver';
CREATE WIDGET TEXT source_sales DEFAULT 'bronze_sales';
CREATE WIDGET TEXT sink_sales DEFAULT 'silver_sales';
CREATE WIDGET TEXT w_sales DEFAULT '1900-01-01';

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')
schema_source = dbutils.widgets.get('schema_source')
schema_sink = dbutils.widgets.get('schema_sink')
source_sales = dbutils.widgets.get('source_sales')
sink_sales = dbutils.widgets.get('sink_sales')
w_sales = dbutils.widgets.get('w_sales')

### Traemos los registros incrementales

In [0]:
source_sales_table = f'{catalog_name}.{schema_source}.{source_sales}'
df_sales = spark.sql(f"SELECT * FROM {source_sales_table} WHERE updated_at > '{w_sales}'")

In [0]:
df_sales_dep = (
    df_sales
    .withColumn('rn', F.row_number().over(Window.partitionBy('sales_id').orderBy(F.col('updated_at').desc())))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

In [0]:
df_sales_dep = (
    df_sales_dep
    .withColumn('sales_subtotal',F.round(F.col('sales_subtotal'),2))
    .withColumn('created_at',F.to_timestamp(F.col('end_at'), 'dd/MM/yyyy H:mm'))
    .withColumn('end_at',F.to_timestamp(F.col('end_at'), 'dd/MM/yyyy H:mm'))
    .withColumn('medio_de_pago', F.initcap(F.trim(F.col('medio_de_pago'))))
    .withColumn('bank_origin', F.initcap(F.trim(F.col('bank_origin'))))
    .withColumn('status', F.lower(F.trim(F.col('status'))))
).drop('_ingestion_timestamp','_source_file')

In [0]:
df_sales_invalid = df_sales_dep.filter(
    (F.col('quantity') < 0) |
    (F.col('sales_subtotal') < 0) |
    (~F.col('status').isin('created','cancelled','finalized')) |
    (F.col('created_at') < '2025-01-01') |
    (F.col('end_at') < F.col('created_at'))
)


invalid_count = df_sales_invalid.count()
if invalid_count > 0:
    print(f"⚠️ {invalid_count} registros inválidos:")

    invalid_condition = (
        (F.col('quantity') < 0) |
        (F.col('sales_subtotal')) < 0 |
        (~F.col('status').isin('created','cancelled','finalized')) |
        (F.col('created_at') < '2025-01-01') |
        (F.col('end_at') < F.col('created_at'))
    )

    df_sales_quarantine = (
        df_sales_invalid
        .withColumn('rejection_reason',
            F.when(F.col('quantity') < 0, 'negative quantity')
            .when(F.col('sales_subtotal') < 0, 'negative subtotal')
            .when(~F.col('status').isin('created', 'cancelled', 'finalized'), 'invalid status')
            .when(F.col('created_at') < '2025-01-01', 'created_at out of range')
            .when(F.col('end_at') < F.col('created_at'), 'end_at before created_at')
            .otherwise('unknown')
        )
        .withColumn('quarantined_at', F.current_timestamp())
    )

    df_sales_quarantine.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(f"{catalog_name}.{schema_sink}.quarantine_sales")

    df_sales_dep = df_sales_dep.filter(~invalid_condition)

else:
    print("✅ Todos los registros son válidos")

In [0]:
target_sales_table = f'{catalog_name}.{schema_sink}.{sink_sales}'
if spark.catalog.tableExists(target_sales_table):

    source_table = DeltaTable.forName(spark, target_sales_table)
    df_sales_dep = df_sales_dep.withColumn('_ingestion_timestamp', F.current_timestamp())

    (
        source_table.alias("target")
        .merge(
            df_sales_dep.alias("source"),
            f"target.sales_id = source.sales_id"
        )
        # Registro existe y cambió algo → actualizar
        .whenMatchedUpdateAll()
        # Registro nuevo → insertar
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    (
        df_sales_dep.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .clusterBy('updated_at','medio_de_pago','sales_subtotal')  # Liquid Clustering
        .saveAsTable(target_sales_table)
    )